In [1]:
rm(list=ls())
BiocManager::install("RUVSeq")
sapply(c("sva", "dplyr", "DESeq2", "ggplot2", "reshape2", "gridExtra", "scales", 
         "RUVSeq", "ggpubr", "BatchQC"), require, character.only=TRUE)

## Parameters (change paths when necessary)
data_dir <- "./"  # path to the signature data (.rds)
source("./combat_seq_file/gfrn_helpers.R")  # path to gfrn_helpers.R
source("./combat_seq_file/ComBat_seq.R"); 
source("./combat_seq_file/helper_seq.R")   
# path to the combat-seq scripts (or use the sva package on github, in which case comment out the above line)

pathway_regex <- c("her2", "^egfr", "kraswt")  
set.seed(1)

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cloud.r-project.org

Bioconductor version 3.18 (BiocManager 1.30.25), R 4.3.3 (2024-02-29)

Warning message:
“package(s) not installed when version(s) same as or greater than current; use
  `force = TRUE` to re-install: 'RUVSeq'”
Installation paths not writeable, unable to update packages
  path: /usr/lib/R/library
  packages:
    class, cluster, foreign, KernSmooth, nlme, nnet, rpart, spatial, survival
  path: /usr/local/lib/R/site-library
  packages:
    askpass, BH, bit, bit64, bitops, broom, commonmark, cpp11, credentials,
    data.table, dotCall64, evaluate, fastDummies, fitdistrplus, FNN,
    fontawesome, fs, future.apply, gert, glue, gplots, gtable, igraph, knitr,
    later, matrixStats, pillar, progressr, promises, Rcpp, reticulate, rlang,
    rmarkdown, rstudioapi, spam, spatstat.data, spatstat.univar,
  

sva     dplyr    DESeq2   ggplot2  reshape2 gridExtra    scales    RUVSeq 
     TRUE      TRUE      TRUE      TRUE      TRUE      TRUE      TRUE      TRUE 
   ggpubr   BatchQC 
    FALSE      TRUE

In [17]:
file_path <- "data_preprocessed/temp_count_matrix_covariates_removal/count_matrix_raw_without_batch_7.csv"

# Read the CSV file into a data frame
count_matrix <- read.csv(file_path, row.names = 1)

# Display the structure of the loaded data frame

count_matrix <- t(count_matrix)

# Display the first few rows of the data frame
head(count_matrix)


,23_120411,591_120522,691_120605,588_120522,604_120523,545_120516,364_120502,705_120605,711_120531,602_120523,⋯,628_120524,587_120522,168_120423,283_120430,721_120531,677_120604,142_120419,120_120418,483_120515,174_120424
ENSG00000101901,471,536,474,494,636,155,340,255,316,663,⋯,146,279,552,288,197,235,334,311,667,307
ENSG00000174839,355,477,460,478,588,218,379,296,298,676,⋯,192,270,333,239,180,245,260,374,432,393
ENSG00000151466,223,299,249,245,446,163,265,204,229,330,⋯,84,190,279,219,94,121,174,255,234,271
ENSG00000168288,313,492,465,464,660,148,281,319,348,642,⋯,131,184,435,283,152,198,301,180,633,266
ENSG00000112343,102,160,191,107,151,139,126,115,143,191,⋯,82,89,50,62,140,159,98,194,102,287
ENSG00000172890,841,916,1193,1028,1154,587,915,721,757,1287,⋯,434,772,892,645,633,576,686,1000,978,826


In [18]:
# Replace "your_file.csv" with the actual path or URL of your CSV file
file_path <- "data_preprocessed/temp_count_matrix_covariates_removal/covariates_clinical_data_AD_NCI_without_batch_7.csv"

# Read the CSV file into a data frame
clinical_data <- read.csv(file_path, row.names = 1)

# Display the structure of the loaded data frame


# Display the first few rows of the data frame
head(clinical_data)

,Batch,educ,cogdx,race,pmi,age_at_visit_max,ceradsc,braaksc,msex,Category
,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
23_120411,6,18,4,1,6.416667,90,1,4,0,AD with Plaques
591_120522,2,12,4,1,5.200000,90,1,4,0,AD with Plaques
691_120605,5,14,1,1,4.833333,90,4,3,0,NCI with No Plaques
588_120522,1,18,1,1,4.400000,90,4,4,0,NCI with No Plaques
604_120523,2,20,4,1,17.833333,90,1,5,0,AD with Plaques
545_120516,4,16,4,1,7.750000,90,1,3,1,AD with Plaques


In [19]:
tail(clinical_data)


,Batch,educ,cogdx,race,pmi,age_at_visit_max,ceradsc,braaksc,msex,Category
,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
721_120531,4,13,4,1,3.000000,72.04381,1,5,0,AD with Plaques
677_120604,4,23,1,1,18.666667,71.72074,4,3,0,NCI with No Plaques
142_120419,6,23,1,1,7.000000,71.60301,4,2,1,NCI with No Plaques
120_120418,5,26,1,1,4.083333,71.48255,4,1,1,NCI with No Plaques
483_120515,1,10,1,1,5.433333,70.51061,4,0,1,NCI with No Plaques
174_120424,5,12,1,1,5.000000,66.95962,4,1,1,NCI with No Plaques


In [20]:
batch_sub <- clinical_data[, 'Batch']

In [21]:
group_sub <- clinical_data[, 'Category']
group_sub

[1] "AD with Plaques"     "AD with Plaques"     "NCI with No Plaques"
  [4] "NCI with No Plaques" "AD with Plaques"     "AD with Plaques"    
  [7] "AD with Plaques"     "AD with Plaques"     "AD with Plaques"    
 [10] "AD with Plaques"     "AD with Plaques"     "AD with Plaques"    
 [13] "AD with Plaques"     "NCI with No Plaques" "AD with Plaques"    
 [16] "NCI with No Plaques" "AD with Plaques"     "AD with Plaques"    
 [19] "AD with Plaques"     "AD with Plaques"     "AD with Plaques"    
 [22] "NCI with No Plaques" "AD with Plaques"     "NCI with No Plaques"
 [25] "AD with Plaques"     "NCI with No Plaques" "AD with Plaques"    
 [28] "AD with Plaques"     "AD with Plaques"     "AD with Plaques"    
 [31] "AD with Plaques"     "AD with Plaques"     "AD with Plaques"    
 [34] "AD with Plaques"     "AD with Plaques"     "AD with Plaques"    
 [37] "AD with Plaques"     "AD with Plaques"     "NCI with No Plaques"
 [40] "AD with Plaques"     "AD with Plaques"     "AD with Plaques"    
 [43] "AD with Plaques"     "AD with Plaques"     "AD with Plaques"    
 [46] "AD with Plaques"     "NCI with No Plaques" "AD with Plaques"    
 [49] "AD with Plaques"     "NCI with No Plaques" "NCI with No Plaques"
 [52] "AD with Plaques"     "NCI with No Plaques" "AD with Plaques"    
 [55] "NCI with No Plaques" "AD with Plaques"     "AD with Plaques"    
 [58] "NCI with No Plaques" "AD with Plaques"     "AD with Plaques"    
 [61] "AD with Plaques"     "NCI with No Plaques" "AD with Plaques"    
 [64] "NCI with No Plaques" "NCI with No Plaques" "AD with Plaques"    
 [67] "AD with Plaques"     "AD with Plaques"     "AD with Plaques"    
 [70] "AD with Plaques"     "NCI with No Plaques" "AD with Plaques"    
 [73] "AD with Plaques"     "NCI with No Plaques" "AD with Plaques"    
 [76] "AD with Plaques"     "AD with Plaques"     "AD with Plaques"    
 [79] "AD with Plaques"     "NCI with No Plaques" "NCI with No Plaques"
 [82] "AD with Plaques"     "AD with Plaques"     "NCI with No Plaques"
 [85] "AD with Plaques"     "AD with Plaques"     "AD with Plaques"    
 [88] "NCI with No Plaques" "AD with Plaques"     "NCI with No Plaques"
 [91] "AD with Plaques"     "NCI with No Plaques" "AD with Plaques"    
 [94] "NCI with No Plaques" "NCI with No Plaques" "AD with Plaques"    
 [97] "AD with Plaques"     "AD with Plaques"     "NCI with No Plaques"
[100] "AD with Plaques"     "NCI with No Plaques" "AD with Plaques"    
[103] "NCI with No Plaques" "NCI with No Plaques" "AD with Plaques"    
[106] "AD with Plaques"     "AD with Plaques"     "NCI with No Plaques"
[109] "NCI with No Plaques" "NCI with No Plaques" "AD with Plaques"    
[112] "NCI with No Plaques" "NCI with No Plaques" "AD with Plaques"    
[115] "AD with Plaques"     "AD with Plaques"     "NCI with No Plaques"
[118] "AD with Plaques"     "NCI with No Plaques" "NCI with No Plaques"
[121] "AD with Plaques"     "NCI with No Plaques" "NCI with No Plaques"
[124] "NCI with No Plaques" "AD with Plaques"     "AD with Plaques"    
[127] "NCI with No Plaques" "AD with Plaques"     "AD with Plaques"    
[130] "AD with Plaques"     "AD with Plaques"     "NCI with No Plaques"
[133] "AD with Plaques"     "NCI with No Plaques" "NCI with No Plaques"
[136] "NCI with No Plaques" "NCI with No Plaques" "AD with Plaques"    
[139] "NCI with No Plaques" "AD with Plaques"     "AD with Plaques"    
[142] "NCI with No Plaques" "NCI with No Plaques" "AD with Plaques"    
[145] "NCI with No Plaques" "AD with Plaques"     "NCI with No Plaques"
[148] "NCI with No Plaques" "AD with Plaques"     "NCI with No Plaques"
[151] "NCI with No Plaques" "NCI with No Plaques" "NCI with No Plaques"
[154] "NCI with No Plaques" "NCI with No Plaques" "NCI with No Plaques"
[157] "NCI with No Plaques" "NCI with No Plaques" "NCI with No Plaques"
[160] "NCI with No Plaques" "AD with Plaques"     "NCI with No Plaques"
[163] "NCI with No Plaques" "NCI with No Plaques" "AD with Plaques"    
[166] "NCI with No Plaques" "NCI with No P

In [22]:
colnames(clinical_data)

[1] "Batch"            "educ"             "cogdx"            "race"            
 [5] "pmi"              "age_at_visit_max" "ceradsc"          "braaksc"         
 [9] "msex"             "Category"

In [23]:
clinical_data

,Batch,educ,cogdx,race,pmi,age_at_visit_max,ceradsc,braaksc,msex,Category
,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
23_120411,6,18,4,1,6.416667,90,1,4,0,AD with Plaques
591_120522,2,12,4,1,5.200000,90,1,4,0,AD with Plaques
691_120605,5,14,1,1,4.833333,90,4,3,0,NCI with No Plaques
588_120522,1,18,1,1,4.400000,90,4,4,0,NCI with No Plaques
604_120523,2,20,4,1,17.833333,90,1,5,0,AD with Plaques
545_120516,4,16,4,1,7.750000,90,1,3,1,AD with Plaques
364_120502,5,16,4,1,19.000000,90,1,5,0,AD with Plaques
705_120605,3,16,4,1,7.083333,90,1,5,0,AD with Plaques
711_120531,3,14,4,1,8.166667,90,1,3,0,AD with Plaques


In [24]:
count_matrix

,23_120411,591_120522,691_120605,588_120522,604_120523,545_120516,364_120502,705_120605,711_120531,602_120523,⋯,628_120524,587_120522,168_120423,283_120430,721_120531,677_120604,142_120419,120_120418,483_120515,174_120424
ENSG00000101901,471,536,474,494,636,155,340,255,316,663,⋯,146,279,552,288,197,235,334,311,667,307
ENSG00000174839,355,477,460,478,588,218,379,296,298,676,⋯,192,270,333,239,180,245,260,374,432,393
ENSG00000151466,223,299,249,245,446,163,265,204,229,330,⋯,84,190,279,219,94,121,174,255,234,271
ENSG00000168288,313,492,465,464,660,148,281,319,348,642,⋯,131,184,435,283,152,198,301,180,633,266
ENSG00000112343,102,160,191,107,151,139,126,115,143,191,⋯,82,89,50,62,140,159,98,194,102,287
ENSG00000172890,841,916,1193,1028,1154,587,915,721,757,1287,⋯,434,772,892,645,633,576,686,1000,978,826
ENSG00000150991,4471,7503,7709,7994,10102,2263,4308,4956,4054,9490,⋯,2074,3038,8187,5401,3103,2979,5570,3104,9291,4231
ENSG00000100029,733,902,778,950,1315,500,686,714,666,1254,⋯,446,553,1108,777,489,433,671,654,1249,652
ENSG00000181894,366,582,454,524,637,202,288,331,262,632,⋯,180,262,604,326,215,186,394,250,754,278
ENSG00000206560,707,951,1019,986,1307,402,672,617,730,1153,⋯,255,560,954,616,439,410,690,735,1171,693


In [25]:
start_time <- Sys.time()
count_matrix_combatseq_sub <- ComBat_seq(counts=count_matrix, batch=batch_sub, group=group_sub, shrink=FALSE)
end_time <- Sys.time()
print(end_time - start_time)

Found 8 batches
Using full model in ComBat-seq.
Adjusting for 1 covariate(s) or covariate level(s)
Estimating dispersions
Fitting the GLM model
Shrinkage off - using GLM estimates for parameters
Adjusting the data
Time difference of 25.04078 secs


In [26]:
count_matrix_combatseq_sub

,23_120411,591_120522,691_120605,588_120522,604_120523,545_120516,364_120502,705_120605,711_120531,602_120523,⋯,628_120524,587_120522,168_120423,283_120430,721_120531,677_120604,142_120419,120_120418,483_120515,174_120424
ENSG00000101901,434,497,530,367,575,227,381,278,348,615,⋯,209,301,497,314,298,345,280,404,547,421
ENSG00000174839,392,392,484,428,485,287,396,333,327,534,⋯,246,289,342,252,231,315,260,417,324,448
ENSG00000151466,216,268,252,233,410,206,259,199,224,290,⋯,138,181,270,217,143,186,167,259,206,278
ENSG00000168288,290,432,508,326,600,236,330,355,387,579,⋯,215,200,420,313,242,319,265,282,486,378
ENSG00000112343,130,157,200,130,146,121,106,103,129,187,⋯,80,79,74,54,122,141,125,198,127,357
ENSG00000172890,755,844,1171,1129,1065,634,920,701,728,1212,⋯,487,760,792,619,679,638,601,1045,949,955
ENSG00000150991,3276,6423,8300,6307,8918,3365,5113,5546,4362,8200,⋯,3230,3102,6952,6055,4289,4550,4328,4818,7626,6132
ENSG00000100029,582,777,947,738,1122,619,818,764,688,1073,⋯,561,549,946,847,607,554,515,833,971,857
ENSG00000181894,302,520,525,374,551,291,350,380,295,547,⋯,262,294,541,373,309,274,324,359,591,404
ENSG00000206560,692,876,1047,821,1289,581,723,634,785,1047,⋯,349,515,960,633,639,576,669,830,994,827


In [27]:
count_matrix_combatseq_sub['ENSG00000000003','23_120411']

[1] 165

In [28]:
file_path <- "data_preprocessed/temp_count_matrix_covariates_removal/count_matrix_after_iteration_without_batch_7_1_batch.csv"
write.csv(count_matrix_combatseq_sub, file = file_path, row.names = TRUE)